# NFL Draft Prediction with CatBoost

This notebook predicts whether a player will be selected in the NFL Draft using physical measurements, athletic test results, school, and position information.

The workflow covers exploratory data analysis, missing-value handling, categorical features, feature engineering, 5-fold stratified cross-validation, CatBoost modeling, feature importance, and submission generation.

## 1. Setup

The dataset is expected in an `input/` folder containing:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

In [ ]:
%pip install -q catboost

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 2025
DATA_DIR = Path("input")

## 2. Load the Data

In [ ]:
train_file = DATA_DIR / "train.csv"
test_file = DATA_DIR / "test.csv"
sample_sub_file = DATA_DIR / "sample_submission.csv"

required_files = [train_file, test_file, sample_sub_file]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Place the competition files inside the input/ folder. Missing: "
        + ", ".join(missing_files)
    )

train = pd.read_csv(train_file)
test = pd.read_csv(test_file)
sample_sub = pd.read_csv(sample_sub_file)

print(f"train: {train.shape}")
print(f"test: {test.shape}")
print(f"sample submission: {sample_sub.shape}")

### Dataset snapshot

In the recorded experiment, the training set contained **2,781 rows and 16 columns**, while the test set contained **696 rows and 15 columns**. The extra training column is the target, `Drafted`.

In [ ]:
train.head()

## 3. Exploratory Data Analysis

In [ ]:
target_distribution = train["Drafted"].value_counts(normalize=True).sort_index() * 100
print(target_distribution.rename("percentage"))

The recorded training split contained approximately **35.17% not drafted** and **64.83% drafted** players.

In [ ]:
missing_summary = pd.DataFrame({
    "train_missing": train.isna().sum(),
    "test_missing": test.isna().sum()
})
missing_summary[missing_summary.sum(axis=1) > 0]

Missing values occur in several physical-test columns, including age, sprint, jump, bench press, agility, and shuttle measurements.

In [ ]:
numeric_cols = train.select_dtypes(include="number").drop(columns=["Id", "Drafted"])

plt.figure(figsize=(12, 10))
sns.heatmap(
    numeric_cols.corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5
)
plt.title("Correlation Matrix of Numeric Features")
plt.tight_layout()
plt.show()

Two notable relationships observed in the original EDA were:

- `Agility_3cone` and `Shuttle`: correlation around **0.88**
- `Sprint_40yd` and `Broad_Jump`: correlation around **-0.82**

The negative sprint relationship is intuitive because a lower 40-yard dash time means better performance, while higher jump distance is better.

## 4. Preprocessing

In [ ]:
# Work on copies so the raw data remains unchanged.
train_model = train.copy()
test_model = test.copy()

# Id is an identifier rather than a predictive feature.
train_model = train_model.drop(columns=["Id"])
test_model = test_model.drop(columns=["Id"])

# Keep School and add a frequency feature instead of dropping it.
school_freq = train_model["School"].value_counts()
train_model["School_freq"] = train_model["School"].map(school_freq)
test_model["School_freq"] = test_model["School"].map(school_freq).fillna(0)

missing_cols = [
    "Age",
    "Sprint_40yd",
    "Vertical_Jump",
    "Bench_Press_Reps",
    "Broad_Jump",
    "Agility_3cone",
    "Shuttle"
]

# Preserve information about originally missing measurements.
for col in missing_cols:
    train_model[f"{col}_missing"] = train_model[col].isna().astype(int)
    test_model[f"{col}_missing"] = test_model[col].isna().astype(int)

# Position-aware mean imputation with a global-mean fallback.
for col in missing_cols:
    position_mean = train_model.groupby("Position")[col].mean()
    global_mean = train_model[col].mean()

    train_model[col] = (
        train_model[col]
        .fillna(train_model["Position"].map(position_mean))
        .fillna(global_mean)
    )

    test_model[col] = (
        test_model[col]
        .fillna(test_model["Position"].map(position_mean))
        .fillna(global_mean)
    )

CatBoost can work directly with categorical features, so `School`, `Player_Type`, `Position_Type`, and `Position` are kept as categorical columns rather than label-encoded.

## 5. CatBoost Baseline

In [ ]:
cat_features = [
    "School",
    "Player_Type",
    "Position_Type",
    "Position"
]

X = train_model.drop(columns=["Drafted"])
y = train_model["Drafted"]

def make_model():
    return CatBoostClassifier(
        iterations=700,
        depth=5,
        learning_rate=0.02,
        l2_leaf_reg=5,
        loss_function="Logloss",
        eval_metric="AUC",
        verbose=0,
        random_seed=RANDOM_STATE
    )

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

auc_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    model = make_model()

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model.fit(X_train, y_train, cat_features=cat_features)

    y_pred = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, y_pred)
    auc_scores.append(auc)

    print(f"Fold {fold}: AUC = {auc:.4f}")

print(f"Average Validation AUC: {np.mean(auc_scores):.4f}")

### Recorded baseline result

The original run of this CatBoost configuration produced a **5-fold average validation ROC-AUC of 0.8359**.

## 6. Feature Engineering

The project adds domain-inspired features that combine body size, speed, explosiveness, agility, and strength.

In [ ]:
def add_engineered_features(df):
    df = df.copy()

    df["BMI"] = df["Weight"] / (df["Height"] ** 2)
    df["Weight_per_Height"] = df["Weight"] / df["Height"]
    df["Speed_Score"] = df["Weight"] / df["Sprint_40yd"]
    df["Explosion_Score"] = df["Vertical_Jump"] * df["Broad_Jump"]
    df["Agility_Score"] = df["Agility_3cone"] + df["Shuttle"]
    df["Strength_to_Weight"] = df["Bench_Press_Reps"] / df["Weight"]

    return df

train_fe = add_engineered_features(train_model)
test_fe = add_engineered_features(test_model)

X_fe = train_fe.drop(columns=["Drafted"])
y = train_fe["Drafted"]

In [ ]:
auc_scores_fe = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_fe, y), start=1):
    model_fe = make_model()

    X_train, X_valid = X_fe.iloc[train_idx], X_fe.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model_fe.fit(X_train, y_train, cat_features=cat_features)

    y_pred = model_fe.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, y_pred)
    auc_scores_fe.append(auc)

    print(f"Fold {fold}: AUC = {auc:.4f}")

print(f"Average Validation AUC: {np.mean(auc_scores_fe):.4f}")

### Recorded feature-engineered result

The original run improved the **5-fold average validation ROC-AUC from 0.8359 to 0.8391** after adding the engineered features.

## 7. Feature Importance

In [ ]:
final_model = make_model()
final_model.fit(X_fe, y, cat_features=cat_features)

feature_importances = pd.DataFrame({
    "Feature": X_fe.columns,
    "Importance": final_model.feature_importances_
}).sort_values("Importance", ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importances, x="Importance", y="Feature")
plt.title("CatBoost Feature Importances")
plt.tight_layout()
plt.show()

feature_importances.head(15)

## 8. Generate Test Predictions

In [ ]:
test_pred_proba_list = []

for train_idx, valid_idx in skf.split(X_fe, y):
    fold_model = make_model()
    fold_model.fit(
        X_fe.iloc[train_idx],
        y.iloc[train_idx],
        cat_features=cat_features
    )

    test_pred_proba_list.append(
        fold_model.predict_proba(test_fe)[:, 1]
    )

test_pred_proba_mean = np.mean(test_pred_proba_list, axis=0)

submission = sample_sub.copy()
submission["Drafted"] = test_pred_proba_mean
submission.to_csv("submission.csv", index=False)

submission.head()

## 9. Next Improvements

Possible extensions include:

- Fold-local preprocessing to make cross-validation even stricter
- Hyperparameter optimization
- Comparing CatBoost with LightGBM or XGBoost
- Position-specific feature engineering
- Probability calibration
- SHAP-based model interpretation
- Repeated cross-validation to test score stability